#STEP 1: Read Bronze Table

In [0]:
from pyspark.sql.functions import col, upper, trim, current_timestamp
from delta.tables import DeltaTable

In [0]:
df = spark.read.table(
    "company_risk_intelligence_platform.bronze.yf_info"
)
df.printSchema()
display(df.limit(5))

# STEP 2 -Transformation Function

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper
)

def transform_yf_info(df):

    df = (
        df
        .withColumn("ticker", upper(trim(col("symbol"))))
        .withColumn("company_name", trim(col("longname")))
        .withColumn("sector", trim(col("sector")))
        .withColumn("industry", trim(col("industry")))
        .withColumn("country", trim(col("country")))
        .withColumn("city", trim(col("city")))
        .drop("file_path")
    )

    # one latest record per company
    df = df.dropDuplicates(["ticker"])

    df = df.select(

        # Company Identity
        "ticker",
        "company_name",
        "shortname",
        "sector",
        "industry",

        # Company Details
        "country",
        "city",
        "address1",
        "zip",
        "website",
        "phone",
        "fulltimeemployees",

        # Governance & Risk
        "overallrisk",
        "auditrisk",
        "boardrisk",
        "compensationrisk",
        "shareholderrightsrisk",

        # Financial Metrics
        "marketcap",
        "enterprisevalue",
        "totalrevenue",
        "ebitda",
        "netincometocommon",
        "freecashflow",
        "totaldebt",
        "totalcash",

        # Market Metrics
        "currentprice",
        "beta",
        "trailingpe",
        "forwardpe",
        "pricetobook",
        "recommendationkey",
        "recommendationmean",
        "targetmeanprice",

        # Business Description
        "longbusinesssummary",

        # Metadata
        "ingestion_ts"
    )

    return df

#SCD Type 1 Merge Function

In [0]:
def scd_merge_table(spark, source_table, target_table, business_key):

    if not spark.catalog.tableExists(target_table):

        print("First Load: Creating Silver Table", target_table)

        source_table.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print("Table Created")

    else:

        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        delta_table.alias("target") \
            .merge(
                source_table.alias("source"),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        print("Merge Successfully Completed")

In [0]:
########################################
######## MAIN LOGIC ####################
########################################

source_table = "company_risk_intelligence_platform.bronze.yf_info"

target_table = "company_risk_intelligence_platform.silver.yf_info"

business_key = ["ticker"]

df = spark.read.table(source_table)

source_table = transform_yf_info(df)

scd_merge_table(
    spark,
    source_table,
    target_table,
    business_key
)